# Setup

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
from dotenv import load_dotenv
from qbstyles import mpl_style
import pytidycensus as tc
from concurrent.futures import ThreadPoolExecutor
from typing import Dict, Literal, List, Union
import glob
import warnings

warnings.filterwarnings("ignore")


%config InlineBackend.figure_format = 'retina'

mpl_style(dark=True)

load_dotenv()

API_KEY: str = os.getenv("CENSUS_KEY")
DATA: str = os.getenv("DATA")

tc.set_census_api_key(API_KEY)

pd.set_option('display.precision', 2)



Census API key has been set for this session.


/tmp/ipykernel_1435398/3053065449.py:7: UserWarning: Mapping functions unavailable due to import error: NameError. To use mapping features, ensure all dependencies are properly installed: pip install pytidycensus[map]
  import pytidycensus as tc


---
---
# Data Collection

## RUCC Codes


### Code Descriptions

|Code|Description|Category|
|---|---|---|
|1|Counties in metro areas of 1 million population or more|Metropolitan|
|2|Counties in metro areas of 250,000 to 1 million population|Metropolitan|
|3|Counties in metro areas of fewer than 250,000 population|Metropolitan|
|4|Urban population of 20,000 or more, adjacent to a metro area|Nonmetropolitan|
|5|Urban population of 20,000 or more, not adjacent to a metro area|Nonmetropolitan|
|6|Urban population of 5,000 to 20,000, adjacent to a metro area|Nonmetropolitan|
|7|Urban population of 5,000 to 20,000, not adjacent to a metro area|Nonmetropolitan|
|8|Urban population of fewer than 5,000, adjacent to a metro area|Nonmetropolitan|
|9|Urban population of fewer than 5,000, not adjacent to a metro area|Nonmetropolitan|


### Variables & Data Processing

- `GEOID`: State & County identifier
- `state`: State abbreviation
- `county`: Name of county
- `RUCC`: RUCC of county

#### Code

In [ ]:
def collect_rucc_data(data_path: str) -> pd.DataFrame:
    data = pd.read_csv(f"{data_path}/RUCC.csv")
    data.drop(columns=["Population_2020", "Description"], inplace=True)
    data.rename(columns={"RUCC_2023": "RUCC"}, inplace=True)
    data["County_Name"] = (
        data["County_Name"].str.replace("County", "").str.strip().str.lower()
    )
    data["County_Name"] = (
        data["County_Name"].str.replace("Parish", "").str.strip().str.lower()
    )

    data.rename(
        columns={"State": "state", "County_Name": "county", "FIPS": "GEOID"},
        inplace=True,
    )

    return data


rucc = collect_rucc_data(data_path=DATA)
rucc.head()


,GEOID,state,county,RUCC
0,1001,AL,autauga,2.0
1,1003,AL,baldwin,3.0
2,1005,AL,barbour,6.0
3,1007,AL,bibb,1.0
4,1009,AL,blount,1.0



# American Community Survey Data

## Overview

- ACS data are collected for three years: 2013, 2018, and 2023.
    - The Census Bureau highly discourages comparing data from years within the same 5-year time period due to the way the 5-year ACS data is collected and constructed, so these are the ideal years to use for a time-series comparison that includes the demographic and socio-economic variables available through the ACS.
- **Race and Total Population Variables**: Individual race variables will be scaled to the total population so that each variable represents the percentage of the population in a county that identifies as each race.
    - Helps reduce the effect of population differences.
- **Median Earnings, Poverty, and Gini Index**: these variables are to reflect the economic conditions in each census tract without going overboard.
    - Total number of people under the poverty level will be scaled by the total population to reflect the poverty rate.
- **Note**: Margin of error columns are dropped becayse pytidycensus handles it at a default of 90% confidence

## Data Processing

- ACS data are collected for three years: 2013, 2018, and 2023.
    - The Census Bureau highly discourages comparing data from years within the same 5-year time period due to the way the 5-year ACS data is collected and constructed, so these are the ideal years to use for a time-series comparison that includes the demographic and socio-economic variables available through the ACS.
- **Race and Total Population Variables**: Individual race variables will be scaled to the total population so that each variable represents the percentage of the population in a county that identifies as each race.
    - Helps reduce the effect of population differences.
- **Median Earnings, Poverty, and Gini Index**: these variables are to reflect the economic conditions in each census tract without going overboard.
    - Total number of people under the poverty level will be scaled by the total population to reflect the poverty rate.
- **Note**: Margin of error columns are dropped becayse pytidycensus handles it at a default of 90% confidence


#### Code

In [ ]:
vars_dict = {
    "poverty": "B17001_002E",
    "white": "B02001_002E",
    "black": "B02001_003E",
    "total_pop": "B01003_001E",
    "am_in_ala_nat": "B02001_004E",
    "asian": "B02001_005E",
    "haw_pac": "B02001_006E",
    "other": "B02001_007E",
    "multi_racial": "B02001_008E",
    "hisp_lat": "B03001_003E",
    "median_earnings": "B08521_001E",
    "gini_index": "B19083_001E",
}


In [ ]:
vars_df = pd.Series(vars_dict).reset_index()
vars_df.columns = ["Variable", "name"]

acs_vars = tc.load_variables(2023, "acs", "acs5", cache=True)
acs_vars[acs_vars["name"].isin(vars_dict.values())]

acs_vars = pd.merge(acs_vars, vars_df, on="name")
acs_vars.drop(["group", "predicateType", "limit", "table"], axis=1, inplace=True)
acs_vars.set_index("Variable", inplace=True)
# acs_vars['label'] = acs_vars['label'].str.replace("Parish", "").str.strip()

acs_vars["label"] = [label[-1] for label in acs_vars["label"].str.split("!")]
acs_vars["label"] = acs_vars["label"].str.replace(":", "").str.strip()

acs_vars


Loaded cached variables for 2023 acs acs5


,name,label,concept
Variable,,,
total_pop,B01003_001E,Total,Total Population
white,B02001_002E,White alone,Race
black,B02001_003E,Black or African American alone,Race
am_in_ala_nat,B02001_004E,American Indian and Alaska Native alone,Race
asian,B02001_005E,Asian alone,Race
haw_pac,B02001_006E,Native Hawaiian and Other Pacific Islander alone,Race
other,B02001_007E,Some Other Race alone,Race
multi_racial,B02001_008E,Two or More Races,Race
hisp_lat,B03001_003E,Hispanic or Latino,Hispanic or Latino Origin by Specific Origin


In [ ]:
states_to_remove = [
    "PR",
    "HI",
    "GU",
    "MP",
    "AS",
    "AK",
    "VI",
]  # Filter states to continental US + D.C.

states = list(
    set([state for state in rucc["State"].values if state not in states_to_remove])
)


In [ ]:
import os
import sys
from contextlib import redirect_stdout


def get_state_data(
    geography: str,
    year: int,
    state: str,
    variables: Dict[str, str],
    geo: bool,
):
    try:
        results = tc.get_acs(
            geography=geography,
            variables=variables,
            state=state,
            year=year,
            geometry=geo,
        )
        results["year"] = year
        cols_to_drop = [col for col in list(census.columns) if "_moe" in col]
        return results.drop(columns=cols_to_drop)

    except Exception as e:
        print(f"{state}, {e}")

    return pd.DataFrame()


# Define a "null" destination
with open(os.devnull, "w") as f:
    with redirect_stdout(f):
        with ThreadPoolExecutor(max_workers=10) as executor:
            futures = []
            for year in [2013, 2018, 2023]:
                for state in states:
                    future = executor.submit(
                        get_state_data,
                        geography="county",
                        state=state,
                        year=year,
                        variables=vars_dict,
                        geo=False,
                    )
                    futures.append(future)

        counties = pd.concat(
            [counties(state=state, cb=True, cache=True) for state in states],
            ignore_index=True,
            sort=False,
        )

census = pd.concat(
    [result.result() for result in futures],
    ignore_index=True,
    sort=False,
)


census.head()


Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS


Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting data from the 2009-2013 5-year ACS
Getting dat

,GEOID,poverty,white,black,total_pop,am_in_ala_nat,asian,haw_pac,other,multi_racial,hisp_lat,median_earnings,gini_index,state,county,NAME,year
0,32001,3619.0,20247,536,24572,1184,840,111,1050,604,3100,31541.0,0.39,32,001,"Churchill County, Nevada",2013
1,32005,4757.0,42189,242,47035,1200,541,63,1344,1456,5312,29940.0,0.45,32,005,"Douglas County, Nevada",2013
2,32027,879.0,5573,251,6752,251,40,3,410,224,1524,42629.0,0.40,32,027,"Pershing County, Nevada",2013
3,32011,248.0,1714,11,1804,41,0,6,25,7,71,72707.0,0.38,32,011,"Eureka County, Nevada",2013
4,32013,2023.0,14795,95,16800,802,134,14,547,413,4143,35525.0,0.41,32,013,"Humboldt County, Nevada",2013


---
## ICE Detention Data

### Overview

From [dataset Github](https://github.com/vera-institute/ice-detention-trends/tree/main#about-the-data)

> Vera’s ICE Detention Trends dashboard reveals an unprecedented level of detail about ICE detention populations—nationally and across the 1,464 facilities in which ICE detained people—on each day of the 16-year period from fiscal year 2009 through the beginning of fiscal year 2026 (October 1, 2008, through October 15, 2025). This repository includes the aggregated data visualized in the dashboard, including information on:
>
> - Midnight population: the daily number of people detained at midnight (nationally and by facility).
> - 24-hour population: the number of people detained for any part of a given day, including those whom ICE transferred or booked-out of custody before midnight (nationally and by facility). While ICE relies solely on midnight populations in its reporting, Vera includes both types of daily populations—midnight and 24-hour—as the two can differ drastically.
> - Book-ins: the daily number of people ICE booked into custody (nationally).
> - Book-outs: the daily number of people ICE booked out of custody (nationally).
> - Facility names, locations, and types (as coded by ICE in other datasets, where available).
> 
> The original datasets included facility names and codes, but no information on location or facility type. Vera drew from additional datasets and public sources to geocode facility > locations and assign facility types. Given the lack of a comprehensive, up-to-date ICE source to assign facility types to all 1,464 facility codes in the dataset, Vera’s categorizations should be interpreted as best-known facility type. To simplify map filtering options, Vera grouped facility types assigned by ICE, as well as ones manually entered by Vera, into the following categories:
>
> - Non-Dedicated: IGSA (Inter-governmental Service Agreement).
> - Dedicated: DIGSA (Dedicated IGSA), SPC (Service Processing Center), CDF (Contract Detention Facility).
> - Federal: USMS IGA (U.S. Marshals Service Inter-governmental Agreement), BOP (Bureau of Prisons), USMS CDF (U.S. Marshals Service Contract Detention Facility), DOD (Department of Defense), MOC (Migrant Operations Center). Because ICE can be added to other federal agencies’ facility contracts or agreements through a “rider,” Vera reports federal facilities as a separate category, rather than grouped with other categories such as non-dedicated facilities.
> - Hold/Staging.
> - Family/Youth: Family, Family Staging, Juvenile. ICE’s use of the “Juvenile” facility type reflects ICE detention and does not refer to facilities used to detain unaccompanied children in the custody of the Office of Refugee Resettlement (ORR).
> - Medical: Facilities coded by ICE as “Hospital” and medical or mental health facilities manually coded by Vera.
> - Hotel: Facilities coded by ICE as “Hotel” and facilities manually coded by Vera.
> - Other/Unknown: Facilities coded by ICE as “Other” or ones for which Vera was unable to assign facility type.


### Facilities Data


> The ICE detention datasets include facility names and codes, but no information on location or facility type. Vera drew from additional datasets and public sources to geocode facility locations and assign facility types. Given the lack of a comprehensive, up-to-date ICE source to assign facility types to all facility codes in the dataset, Vera’s categorizations should be interpreted as best-known facility type.


|Variable|Type|Description|
|---|---|---|
|detention_facility_code|`string`|The unique identifier used in the ICE detention data for each facility|
|detention_facility_name|`string`|The facility name associated with the detention_facility_code in the ICE detention data|
|latitude|`numeric`|The latitude coordinate of the facility location|
|longitude|`numeric`|The longitude coordinate of the facility location|
|address|`string`|The best known facility address|
|city|`string`|The city in which the facility is located|
|county|`string`|The county in which the facility is located|
|state|`string`|The state abbreviation code. This also includes codes for U.S. territories (e.g. "PR" for "Puerto Rico") and Cuba ("CU") for facilities at Naval Station Guantanamo Bay.|
|aor|`aor`|The ICE Area of Responsibility (AOR), originally mapped by Will Craft of the Guardian US. This reflects county boundaries extracted from ICE's [field office map](https://www.ice.gov/doclib/about/offices/ero/pdf/eroFieldOffices.pdf), last updated by ICE in February 2024.|


### Incarceration Data



Facility-level population statistics for each day between October 1, 2008, and October 15, 2025, including midnight population and 24-hour population. 


|Variable|Type|Description|
|---|---|---|
|detention_facility_code|`string`|The unique identifier used in the ICE detention data for each facility|
|detention_facility_name|`string`|The facility name associated with the detention_facility_code in the ICE detention data|
|state|`string`|The state abbreviation code. This also includes codes for U.S. territories (e.g. "PR" for "Puerto Rico") and Cuba ("CU") for facilities at Naval Station Guantanamo Bay.|
|date|`date`|The day each count is reported for (`yyyy-mm-dd` format)|
|daily_pop|`numeric`|24-hour population: the number of people detained for any part of a given day, including those whom ICE transferred or booked-out of custody before midnight|
|midnight_pop|`numeric`|Midnight population: the daily number of people detained at midnight|


In [ ]:
ice_dir = f"{DATA}/ice-detention-trends/facilities/by_state"

file_pattern = os.path.join(ice_dir, "*.csv")
csvs = glob.glob(file_pattern)
ice_data = pd.concat([pd.read_csv(csv) for csv in csvs], ignore_index=True, sort=False)
ice_data["date"] = pd.to_datetime(ice_data["date"])
ice_data = ice_data.loc[ice_data["date"] > "2009"]
# ice_data["quarter"] = ice_data["date"].dt.quarter
ice_data["year"] = ice_data["date"].dt.year
ice_data = ice_data[ice_data["year"].isin([2013, 2018, 2023])]

if "facilities" not in locals():
    print("Processing facilities data")
    facilities = pd.read_csv(f"{DATA}/ice-detention-trends/metadata/facilities.csv")
    facilities.drop(columns=["address", "city", "zip"], inplace=True)

    facilities.rename(columns={"FIPS": "GEOID"}, inplace=True)
    facilities.drop(columns=["detention_facility_name", "state"], inplace=True)
    facilities.sort_values("county", inplace=True)
else:
    print("facilities data already processed")

ice_data = pd.merge(
    facilities,
    ice_data,
    on="detention_facility_code",
    how="left",
)
ice_data = ice_data[ice_data["year"].isin([2013, 2018, 2023])]
ice_data.dropna(subset=["county"], inplace=True)

cols = [
    "county",
    "state",
    "daily_pop",
    "midnight_pop",
    "year",
]

ice_data = ice_data[cols]

ice_data = ice_data.groupby(["year", "state", "county"]).sum().reset_index()
ice_data = ice_data.loc[~ice_data["state"].isin(states_to_remove)]
ice_data["county"] = ice_data["county"].str.strip().str.lower()


## Incarceration Trends Dataset

See [Incarceration Trends Dataset](https://github.com/vera-institute/incarceration-trends)

> Vera’s Incarceration Trends Project (ITP) dataset provides county-level data on prison and jail incarceration and related measures over time for the entire United States.
> 
> Vera assembled this dataset primarily using information collected by the U.S. Department of Justice Bureau of Justice Statistics (BJS) and supplemented it with data collected by Vera directly from state and local agencies
when federal data was not available. The BJS data used in the ITP dataset run through 2019 for prisons and through 2023 for jails.




### Variable Descriptions for Incarceration Metrics

Due to rapid changes in the jail populations in recent years and more broadly available data on jails, Vera now produces jail population estimate (total_jail_pop) on a quarterly basis. 

Generally, this means Vera aims to measure the number of people in jail on or near these dates:
• Q1: March 31,
• Q2: June 30,
• Q3: September 30, and
• Q4: December 31.